# End-to-End Pipeline: Combined Generator -> RR_5 Attention Reranker -> Final K

Runs the combined candidate generator and the `RR_5_generated_candidates` reranker as one
pipeline:

- **Candidate generation** (`CG_content_two_tower_rolling_popularity.ipynb`): 500 candidates
  per impression. Ranks 1-200 are rolling prior-hour popularity; ranks 201-500 are the
  content two-tower's personalised top 300.
- **Reranking** (`RR_5_generated_candidates.ipynb`): the `+metadata` attention reranker trained
  on the generator's training slates orders those 500 candidates.

The reranker's final Top-K list is evaluated with the same metrics as the baseline pipeline:
every metric in `cg_evaluation.py` at `M = K`, plus MRR@K and nDCG@K from `rr_evaluation.py`.
`generator_order` (the generator's own cutoff order) is kept as a reference row.

**Labels are incomplete.** A retrieved article that was never shown has unknown relevance: it
counts as "not a known click", not as a negative. Every known click stays in the denominator,
so clicks the generator missed count as misses.

The notebook reads the RR_5 validation/test score file when it exists. If that file is missing
or does not cover the selected candidates, it falls back to scoring with the saved RR_5
checkpoint and a causal feature replay, so it can run on its own from the candidate file.


## 1.1 Load Packages


In [1]:
import json
import os
import sys
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import polars as pl
import pyarrow as pa
import pyarrow.parquet as pq
import torch
import torch.nn.functional as F
from IPython.display import display
from torch import nn


## 1.2 Set Up Paths

`NEWSREC_ROOT` overrides the project checkout. `NEWSREC_DATA_ROOT` can point at a separate
data root; otherwise the checkout is used. The candidate folder and reranker folder are
resolved against both roots so the notebook works from the normal checkout and from a copied
data root.

For a test run, both stage artefacts must exist:

- `recommendations_test_top500.parquet` from the CG notebook with `TARGET_SPLIT = "test"`.
- `generated_test_scores.parquet` from the RR_5 test section. If it is missing, the fallback
  scorer in this notebook builds the test features and scores them.

`RR5_RUN_TAG` selects which RR_5 run to consume. The default `None` reads the legacy
`branch_interaction_generated` folder, which holds the 5,000-training-impression run. To use
a different training sample, set `TRAIN_IMPRESSIONS` and `RUN_TAG` in
`RR_5_generated_candidates.ipynb`, rerun it, then set the same tag here. Each tagged run
keeps its own checkpoint, preprocessor, replay cache and score files.


In [2]:
CODE_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run_all.py").is_file()
)
ROOT = Path(os.environ.get("NEWSREC_ROOT", CODE_ROOT)).expanduser().resolve()
REQUESTED_DATA_ROOT = Path(os.environ.get("NEWSREC_DATA_ROOT", ROOT)).expanduser().resolve()
REPLAY_ROOT = (
    REQUESTED_DATA_ROOT
    if (REQUESTED_DATA_ROOT / "1_Data_Preparation" / "processed").exists()
    else ROOT
)

PROCESSED = REPLAY_ROOT / "1_Data_Preparation" / "processed"
REPRESENTATIONS = REPLAY_ROOT / "3_Representations" / "artifacts"
RUN = REPLAY_ROOT / "4_Feature_Engineering" / "artifacts" / "prequential-all"
OUTPUT_DIR = REPLAY_ROOT / "5_Models" / "5_03_Pipeline_Evaluation" / "artifacts"
RESULTS = REPLAY_ROOT / "results"
REPLAY_CACHE_DIR = OUTPUT_DIR / "replayed_features"


def first_existing(*paths):
    for path in paths:
        if path.exists():
            return path
    return paths[0]


CANDIDATE_ARTIFACTS = first_existing(
    ROOT / "5_Models" / "5_01_Candidate_Generation" / "artifacts"
    / "combined_content_two_tower_rolling_popularity",
    REPLAY_ROOT / "5_Models" / "5_01_Candidate_Generation" / "artifacts"
    / "combined_content_two_tower_rolling_popularity",
)
RR5_RUN_TAG = None  # None = legacy branch_interaction_generated (5k-impression run)
RERANKER_BASE = first_existing(
    REPLAY_ROOT / "5_Models" / "5_02_Reranking" / "artifacts" / "branch_interaction_generated",
    ROOT / "5_Models" / "5_02_Reranking" / "artifacts" / "branch_interaction_generated",
)
RERANKER_ARTIFACTS = RERANKER_BASE / RR5_RUN_TAG if RR5_RUN_TAG else RERANKER_BASE
RERANKER_DIR = (
    RERANKER_ARTIFACTS
    / "branch_plus_metadata__softmax_plus_lambda_ndcg__generated"
)
if not RERANKER_DIR.exists():
    raise FileNotFoundError(
        f"RR_5 run not found: {RERANKER_DIR}. Set RR5_RUN_TAG to an existing run tag, "
        f"or run RR_5_generated_candidates.ipynb with RUN_TAG='{RR5_RUN_TAG}'."
    )

sys.path.insert(0, str(ROOT / "5_Models" / "5_01_Candidate_Generation"))
sys.path.insert(0, str(ROOT / "5_Models" / "5_02_Reranking"))
sys.path.insert(0, str(ROOT / "src"))

from cg_evaluation import CandidateEvaluator
from newsrec.common import DEFAULT_SETTINGS
from newsrec.pipeline import replay_batches, time_horizon
from newsrec.ranking import TemporalState, build_features, feature_schema
from newsrec.representations import Bundle
from rr_evaluation import evaluate_retrieved_candidates

for path in (OUTPUT_DIR, RESULTS, REPLAY_CACHE_DIR):
    path.mkdir(parents=True, exist_ok=True)

print("Repository:", ROOT)
print("Replay root:", REPLAY_ROOT)
print("Candidate folder:", CANDIDATE_ARTIFACTS)
print("RR_5 run tag:", RR5_RUN_TAG)
print("RR_5 run root:", RERANKER_ARTIFACTS)
print("RR_5 model folder:", RERANKER_DIR)


Repository: C:\Users\Yong Rui\BT4222-Group-6-
Replay root: C:\Users\Yong Rui\BT4222-Group-6-
Candidate folder: C:\Users\Yong Rui\BT4222-Group-6-\5_Models\5_01_Candidate_Generation\artifacts\combined_content_two_tower_rolling_popularity
RR_5 run tag: None
RR_5 run root: C:\Users\Yong Rui\BT4222-Group-6-\5_Models\5_02_Reranking\artifacts\branch_interaction_generated
RR_5 model folder: C:\Users\Yong Rui\BT4222-Group-6-\5_Models\5_02_Reranking\artifacts\branch_interaction_generated\branch_plus_metadata__softmax_plus_lambda_ndcg__generated


## 1.3 Configuration


In [3]:
TARGET_SPLIT = "validation"  # "validation" or "test"
K = 10
EVAL_IMPRESSIONS = 10_000  # None = every impression in the candidate file
SEED = 4222
CANDIDATE_CAP = 500
REUSE_SCORES = True
RESUME = True

CANDIDATE_PATH = (
    CANDIDATE_ARTIFACTS / f"recommendations_{TARGET_SPLIT}_top{CANDIDATE_CAP}.parquet"
)
RERANKER_SCORE_PATH = RERANKER_DIR / f"generated_{TARGET_SPLIT}_scores.parquet"
IMPRESSION_TAG = "all" if EVAL_IMPRESSIONS is None else f"{EVAL_IMPRESSIONS}imp"
BASE_MODEL_NAME = "attention_reranker_generated"

# RR_5 model settings (must match the saved checkpoint)
MODEL_DIM = 64
ATTENTION_HEADS = 2
ATTENTION_DROPOUT = 0.10
HIDDEN_DIMS = (256, 64)
DROPOUT = 0.20
HISTORY_CAP = 30
SHORT_TERM_CAP = 10
SCORING_BATCH_ROWS = 4096
DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

schema = json.loads((RUN / "feature_schema.json").read_text())
FEATURES = schema["model_columns"]
if feature_schema(DEFAULT_SETTINGS)["model_columns"] != FEATURES:
    raise ValueError("The feature code and the saved feature run disagree on the schema")

print("Target split:", TARGET_SPLIT)
print("Final K:", K)
print("Evaluation impressions:", EVAL_IMPRESSIONS)
print("Candidate file:", CANDIDATE_PATH)
print("RR_5 score file:", RERANKER_SCORE_PATH)


Target split: validation
Final K: 10
Evaluation impressions: 10000
Candidate file: C:\Users\Yong Rui\BT4222-Group-6-\5_Models\5_01_Candidate_Generation\artifacts\combined_content_two_tower_rolling_popularity\recommendations_validation_top500.parquet
RR_5 score file: C:\Users\Yong Rui\BT4222-Group-6-\5_Models\5_02_Reranking\artifacts\branch_interaction_generated\branch_plus_metadata__softmax_plus_lambda_ndcg__generated\generated_validation_scores.parquet


## 1.4 Load Generated Candidates

The combined generator writes exactly 500 unique candidates per impression: cutoffs 1-200 are
popularity and 201-500 are content. Impressions are sampled with the same seed and the same
sorted-unique-id logic as `PE_baseline_pipeline.ipynb`, so the two pipeline result tables use
the same validation requests.


In [4]:
if not CANDIDATE_PATH.exists():
    raise FileNotFoundError(
        f"Candidate file not found: {CANDIDATE_PATH}. Run "
        "CG_content_two_tower_rolling_popularity.ipynb with "
        f"TARGET_SPLIT='{TARGET_SPLIT}' first."
    )

candidates = (
    pl.read_parquet(CANDIDATE_PATH, columns=["impression_uid", "cutoff", "news_code"])
    .with_columns(pl.col("news_code").cast(pl.Int64))
    .sort(["impression_uid", "cutoff"])
)
if candidates.select(["impression_uid", "news_code"]).n_unique() != candidates.height:
    raise ValueError("Candidate file contains duplicate impression/news pairs")

shape = candidates.group_by("impression_uid").agg(
    pl.len().alias("rows"),
    pl.col("cutoff").min().alias("first_cutoff"),
    pl.col("cutoff").max().alias("last_cutoff"),
    pl.col("cutoff").n_unique().alias("cutoffs"),
)
bad = shape.filter(
    (pl.col("rows") != CANDIDATE_CAP)
    | (pl.col("first_cutoff") != 1)
    | (pl.col("last_cutoff") != CANDIDATE_CAP)
    | (pl.col("cutoffs") != CANDIDATE_CAP)
)
if bad.height:
    raise ValueError(f"{bad.height} impressions have an invalid candidate shape")
if K > CANDIDATE_CAP:
    raise ValueError("K cannot exceed the candidate cap")

all_uids = candidates["impression_uid"].unique().sort().to_numpy()
if EVAL_IMPRESSIONS is not None:
    chosen = np.random.default_rng(SEED).choice(
        all_uids, size=min(EVAL_IMPRESSIONS, len(all_uids)), replace=False
    )
    candidates = candidates.filter(pl.col("impression_uid").is_in(np.sort(chosen).tolist()))

selected_uids = candidates["impression_uid"].unique().sort()
n_selected = selected_uids.len()
print(f"Impressions: {n_selected:,} of {len(all_uids):,}")
print(f"Candidate rows: {candidates.height:,}")
print(
    "Rows by generator half:",
    candidates.group_by((pl.col("cutoff") <= 200).alias("popularity")).agg(
        pl.len().alias("rows")
    ).sort("popularity"),
)
display(candidates.head(5))


Impressions: 10,000 of 31,624
Candidate rows: 5,000,000
Rows by generator half: shape: (2, 2)
┌────────────┬─────────┐
│ popularity ┆ rows    │
│ ---        ┆ ---     │
│ bool       ┆ u32     │
╞════════════╪═════════╡
│ false      ┆ 3000000 │
│ true       ┆ 2000000 │
└────────────┴─────────┘


impression_uid,cutoff,news_code
i64,i16,i64
10000019,1,287
10000019,2,51048
10000019,3,39010
10000019,4,59673
10000019,5,34876


## 2.1 Load the RR_5 Reranker Scores

`RR_5_generated_candidates.ipynb` writes `generated_<split>_scores.parquet` for every
generator candidate. The file is reused when it covers every selected (impression,
candidate) pair. Scores are monotone reranker probabilities, so only their order matters.


## 2.2 Fallback: Score with the RR_5 Checkpoint

This cell is only executed when the saved score file is missing or incomplete. It is
self-contained: it loads the RR_5 checkpoint and tabular preprocessor, builds the 50 causal
features for the selected candidates (reusing an existing replay cache when possible), and
scores every pair pointwise. The model code matches `RR_5_generated_candidates.ipynb`.

The replayed-feature cache is looked up in this notebook's artifact folder first and then in
the RR_5 generated-candidate artifact folder. The latter already contains
`validation_top500_10000imp_5000000rows_replay.parquet` and the full validation cache, so a
normal validation run does not rebuild them.


In [5]:
class ArticleInputs:
    def __init__(self, article_index, text, entity, title_tokens):
        self.news_to_row = dict(
            zip(article_index["news_code"].to_list(), article_index["article_row"].to_list())
        )
        self.news_to_row = {int(k): int(v) for k, v in self.news_to_row.items()}
        self.text = text
        self.entity = entity
        self.title_tokens = title_tokens


def load_article_inputs():
    article_index = pl.read_parquet(
        REPRESENTATIONS / "article_index.parquet",
        columns=["news_code", "article_row", "category_code", "subcategory_code"],
    )
    text = np.array(np.load(REPRESENTATIONS / "text64.npy", mmap_mode="r"), dtype=np.float32)
    entity = np.array(
        np.load(REPRESENTATIONS / "article_entity100.npy", mmap_mode="r"),
        dtype=np.float32,
    )
    title_tokens = np.load(REPRESENTATIONS / "title_token_ids.npy").astype(np.int64)
    inputs = ArticleInputs(article_index, text, entity, title_tokens)
    inputs.article_category = np.zeros(text.shape[0], dtype=np.int64)
    inputs.article_subcategory = np.zeros(text.shape[0], dtype=np.int64)
    inputs.article_category[article_index["article_row"].to_numpy()] = article_index[
        "category_code"
    ].to_numpy()
    inputs.article_subcategory[article_index["article_row"].to_numpy()] = article_index[
        "subcategory_code"
    ].to_numpy()
    return inputs


class AdditiveAttention(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.proj = nn.Linear(dim, dim)
        self.query = nn.Parameter(torch.randn(dim) * 0.02)

    def forward(self, states, padding_mask):
        all_padded = padding_mask.all(dim=1)
        safe_mask = padding_mask.clone()
        safe_mask[all_padded, 0] = False
        scores = torch.tanh(self.proj(states)) @ self.query
        weights = torch.softmax(scores.masked_fill(safe_mask, -1e9), dim=1)
        weights = weights.masked_fill(padding_mask, 0.0)
        weights = weights / weights.sum(dim=1, keepdim=True).clamp(min=1e-8)
        return (weights.unsqueeze(-1) * states).sum(dim=1)


class NewsEncoder(nn.Module):
    def __init__(self, config, inputs):
        super().__init__()
        self.config = config
        self.register_buffer(
            "text", torch.from_numpy(inputs.text), persistent=False
        )
        width = self.text.shape[1]

        if config["metadata"]:
            self.register_buffer(
                "entity", torch.from_numpy(inputs.entity), persistent=False
            )
            self.register_buffer(
                "category", torch.from_numpy(inputs.article_category), persistent=False
            )
            self.register_buffer(
                "subcategory",
                torch.from_numpy(inputs.article_subcategory),
                persistent=False,
            )
            self.category_embedding = nn.Embedding(
                int(inputs.article_category.max()) + 1, 16, padding_idx=0
            )
            self.subcategory_embedding = nn.Embedding(
                int(inputs.article_subcategory.max()) + 1, 16, padding_idx=0
            )
            self.entity_projection = nn.Linear(self.entity.shape[1], 32)
            width += 16 + 16 + 32

        if config["title_encoder"]:
            self.register_buffer(
                "title_tokens", torch.from_numpy(inputs.title_tokens), persistent=False
            )
            self.word_embedding = nn.Embedding(
                int(inputs.title_tokens.max()) + 1, MODEL_DIM, padding_idx=0
            )
            self.word_attention = nn.MultiheadAttention(
                MODEL_DIM,
                ATTENTION_HEADS,
                dropout=ATTENTION_DROPOUT,
                batch_first=True,
            )
            self.word_pool = AdditiveAttention(MODEL_DIM)
            width += MODEL_DIM

        self.output = nn.Sequential(nn.Linear(width, MODEL_DIM), nn.LayerNorm(MODEL_DIM))

    def encode_titles(self, rows):
        tokens = self.title_tokens[rows]
        padding = tokens.eq(0)
        padding[padding.all(dim=1), 0] = False
        words = self.word_embedding(tokens)
        attended, _ = self.word_attention(
            words, words, words, key_padding_mask=padding, need_weights=False
        )
        return self.word_pool(attended, padding)

    def forward(self, rows):
        unique_rows, inverse = torch.unique(rows, return_inverse=True)
        parts = [self.text[unique_rows]]
        if self.config["metadata"]:
            parts += [
                self.category_embedding(self.category[unique_rows]),
                self.subcategory_embedding(self.subcategory[unique_rows]),
                self.entity_projection(self.entity[unique_rows]),
            ]
        if self.config["title_encoder"]:
            parts.append(self.encode_titles(unique_rows))
        return self.output(torch.cat(parts, dim=-1))[inverse]


class UserEncoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        if config["position"]:
            self.position_embedding = nn.Embedding(HISTORY_CAP, MODEL_DIM)
        self.self_attention = nn.MultiheadAttention(
            MODEL_DIM,
            ATTENTION_HEADS,
            dropout=ATTENTION_DROPOUT,
            batch_first=True,
        )
        self.norm1 = nn.LayerNorm(MODEL_DIM)
        self.feed_forward = nn.Sequential(
            nn.Linear(MODEL_DIM, 4 * MODEL_DIM),
            nn.ReLU(),
            nn.Dropout(ATTENTION_DROPOUT),
            nn.Linear(4 * MODEL_DIM, MODEL_DIM),
        )
        self.norm2 = nn.LayerNorm(MODEL_DIM)
        self.pool = AdditiveAttention(MODEL_DIM)
        self.default_user = nn.Parameter(torch.zeros(MODEL_DIM))
        if config["short_term"]:
            self.short_term = nn.GRU(MODEL_DIM, MODEL_DIM, batch_first=True)
            self.gate = nn.Linear(2 * MODEL_DIM, MODEL_DIM)
        self.last_gate = None

    @property
    def output_dim(self):
        return MODEL_DIM * (2 if self.config["candidate_pooling"] else 1)

    def forward(self, history, padding_mask, candidates):
        empty = padding_mask.all(dim=1)
        safe_mask = padding_mask.clone()
        safe_mask[empty, -1] = False

        if self.config["position"]:
            ranks = torch.arange(HISTORY_CAP - 1, -1, -1, device=history.device)
            history = history + self.position_embedding(ranks)

        attended, _ = self.self_attention(
            history, history, history, key_padding_mask=safe_mask, need_weights=False
        )
        states = self.norm1(history + attended)
        states = self.norm2(states + self.feed_forward(states))
        states = states.masked_fill(padding_mask.unsqueeze(-1), 0.0)

        user = self.pool(states, padding_mask)
        if self.config["short_term"]:
            recent = states[:, -SHORT_TERM_CAP:]
            _, short = self.short_term(recent)
            short = short.squeeze(0)
            gate = torch.sigmoid(self.gate(torch.cat([user, short], dim=1)))
            user = gate * user + (1 - gate) * short
            self.last_gate = gate.detach().mean().item()
        user = torch.where(empty.unsqueeze(1), self.default_user.unsqueeze(0), user)
        user = user.unsqueeze(1).expand(-1, candidates.shape[1], -1)

        if not self.config["candidate_pooling"]:
            return user
        scores = candidates @ states.transpose(1, 2) / np.sqrt(MODEL_DIM)
        scores = scores.masked_fill(safe_mask.unsqueeze(1), -1e9)
        weights = torch.softmax(scores, dim=-1).masked_fill(
            padding_mask.unsqueeze(1), 0.0
        )
        candidate_user = weights @ states
        return torch.cat([user, candidate_user], dim=-1)


class BranchReranker(nn.Module):
    def __init__(self, config, tabular_width, inputs):
        super().__init__()
        if config["user_encoder"] and not config["text"]:
            raise ValueError("The user encoder needs the news encoder (text=True)")
        if not (config["tabular"] or config["text"]):
            raise ValueError("At least one of the tabular or text branches must be on")
        self.config = config
        width = tabular_width if config["tabular"] else 0
        if config["text"]:
            self.news_encoder = NewsEncoder(config, inputs)
            width += MODEL_DIM
        if config["user_encoder"]:
            self.user_encoder = UserEncoder(config)
            width += self.user_encoder.output_dim

        layers = []
        for hidden in HIDDEN_DIMS:
            layers.extend([nn.Linear(width, hidden), nn.ReLU(), nn.Dropout(DROPOUT)])
            width = hidden
        layers.append(nn.Linear(width, 1))
        self.scorer = nn.Sequential(*layers)

    def forward(self, tabular, candidate_rows, history_rows):
        parts = [tabular] if self.config["tabular"] else []
        if self.config["text"]:
            candidates = self.news_encoder(candidate_rows)
            parts.append(candidates)
            if self.config["user_encoder"]:
                history = self.news_encoder(history_rows)
                parts.append(
                    self.user_encoder(history, history_rows.eq(0), candidates)
                )
        return self.scorer(torch.cat(parts, dim=-1)).squeeze(-1)


def load_checkpoint(path, inputs):
    checkpoint = torch.load(path, map_location=DEVICE, weights_only=False)
    model = BranchReranker(
        checkpoint["config"], checkpoint["tabular_width"], inputs
    ).to(DEVICE)
    model.load_state_dict(checkpoint["state_dict"])
    model.eval()
    return checkpoint, model


def load_profiles(split, impression_ids=None):
    profiles = pl.scan_parquet(
        RUN / f"request_profiles_{split}" / "*.parquet"
    ).select("impression_uid", "full_history_rows")
    if impression_ids is not None:
        profiles = profiles.filter(
            pl.col("impression_uid").is_in(list(impression_ids))
        )
    profiles = profiles.collect()
    return dict(
        zip(profiles["impression_uid"].to_list(), profiles["full_history_rows"].to_list())
    )


def encode_histories(impression_ids, profiles):
    histories = np.zeros((len(impression_ids), HISTORY_CAP), dtype=np.int64)
    for index, uid in enumerate(impression_ids):
        history = profiles[uid][-HISTORY_CAP:]
        if len(history):
            histories[index, -len(history):] = history
    return histories


def replay_candidate_features(split, candidates_frame):
    impression_count = candidates_frame["impression_uid"].n_unique()
    cache_name = (
        f"{split}_top500_{impression_count}imp_"
        f"{candidates_frame.height}rows_replay.parquet"
    )
    for folder in (REPLAY_CACHE_DIR, RERANKER_ARTIFACTS / "replayed_features"):
        cached = folder / cache_name
        if RESUME and cached.exists():
            print("Reusing replayed features:", cached)
            return cached

    frame = (
        candidates_frame.sort(["impression_uid", "cutoff"])
        .to_pandas()
        .reset_index(drop=True)
    )
    rows_by_impression = frame.groupby("impression_uid", sort=False).indices
    target_ids = set(rows_by_impression)
    replay_splits = tuple(
        ("train", "validation", "test")[
            : ("train", "validation", "test").index(split) + 1
        ]
    )

    bundle = Bundle(REPLAY_ROOT, DEFAULT_SETTINGS)
    state = TemporalState(bundle, *time_horizon(REPLAY_ROOT), DEFAULT_SETTINGS)
    cache_path = REPLAY_CACHE_DIR / cache_name
    partial_path = cache_path.with_suffix(".partial.parquet")
    writer, pending, impressions, started = None, [], 0, time.monotonic()

    def flush():
        nonlocal writer, pending
        if not pending:
            return
        out = pd.concat(pending, ignore_index=True)
        out = out[["impression_uid", "news_code", "cutoff", *FEATURES]]
        table = pa.Table.from_pandas(out, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(partial_path, table.schema, compression="zstd")
        writer.write_table(table)
        pending = []

    print(
        f"Replaying {split}: {len(frame):,} candidates for "
        f"{len(rows_by_impression):,} impressions"
    )
    try:
        for timestamp, requests, events in replay_batches(
            REPLAY_ROOT, DEFAULT_SETTINGS, splits=replay_splits
        ):
            state.begin(requests, timestamp)
            selected = requests[
                requests["split"].eq(split)
                & requests["impression_uid"].isin(target_ids)
            ]
            if len(selected):
                pairs = frame.iloc[
                    np.concatenate(
                        [rows_by_impression[uid] for uid in selected["impression_uid"]]
                    )
                ]
                features = build_features(
                    selected, pairs, state, bundle, DEFAULT_SETTINGS, "retrieved"
                )
                pending.append(
                    features.merge(
                        pairs,
                        on=["impression_uid", "news_code"],
                        validate="one_to_one",
                    )
                )
                impressions += len(selected)
                if sum(len(chunk) for chunk in pending) >= 50_000:
                    flush()
                    print(
                        f"  {impressions:,}/{len(rows_by_impression):,} impressions "
                        f"({(time.monotonic() - started) / 60:.1f} min)",
                        flush=True,
                    )
            state.finish(requests, events, timestamp, set(replay_splits))
        flush()
    finally:
        if writer is not None:
            writer.close()
        state.close()

    if impressions != len(rows_by_impression):
        raise ValueError(
            f"Replayed {impressions:,} impressions but "
            f"{len(rows_by_impression):,} were selected"
        )
    partial_path.replace(cache_path)
    print("Saved replayed features:", cache_path)
    return cache_path


def score_candidates(split, candidate_frame):
    inputs = load_article_inputs()
    checkpoint, model = load_checkpoint(RERANKER_DIR / "model.pt", inputs)
    preprocessor = joblib.load(RERANKER_DIR.parent / "preprocessor_generated.joblib")

    features_path = replay_candidate_features(split, candidate_frame)
    replayed = (
        pl.read_parquet(
            features_path, columns=["impression_uid", "news_code", "cutoff", *FEATURES]
        )
        .join(
            candidate_frame.select("impression_uid", "news_code", "cutoff").with_columns(
                pl.col("cutoff").cast(pl.Int16)
            ),
            on=["impression_uid", "news_code", "cutoff"],
            how="inner",
        )
        .sort(["impression_uid", "cutoff"])
    )
    if replayed.height != candidate_frame.height:
        raise ValueError(
            "The replay cache does not cover every selected candidate; "
            "delete the matching cache file and rerun."
        )

    profiles = load_profiles(split, candidate_frame["impression_uid"].unique().to_list())
    uids = replayed["impression_uid"].unique().to_list()
    histories = encode_histories(uids, profiles)
    uid_index = {uid: index for index, uid in enumerate(uids)}
    history_index = np.fromiter(
        (uid_index[uid] for uid in replayed["impression_uid"].to_list()),
        dtype=np.int64,
        count=replayed.height,
    )
    tabular = preprocessor.transform(
        replayed.select(FEATURES).to_pandas()
    ).astype(np.float32, copy=False)
    news_codes = replayed["news_code"].to_list()
    missing_codes = [code for code in news_codes if code not in inputs.news_to_row]
    if missing_codes:
        raise ValueError(
            f"{len(missing_codes)} candidates are missing article rows, e.g. "
            f"{missing_codes[:5]}"
        )
    candidate_rows = np.fromiter(
        (inputs.news_to_row[code] for code in news_codes),
        dtype=np.int64,
        count=len(news_codes),
    )

    scores = []
    with torch.no_grad():
        for start in range(0, replayed.height, SCORING_BATCH_ROWS):
            stop = min(start + SCORING_BATCH_ROWS, replayed.height)
            logits = model(
                torch.from_numpy(tabular[start:stop]).unsqueeze(1).to(DEVICE),
                torch.from_numpy(candidate_rows[start:stop]).unsqueeze(1).to(DEVICE),
                torch.from_numpy(histories[history_index[start:stop]]).to(DEVICE),
            )
            scores.append(torch.sigmoid(logits.squeeze(1)).cpu().numpy())

    scored = replayed.select("impression_uid", "news_code", "cutoff").with_columns(
        pl.Series("score", np.concatenate(scores), dtype=pl.Float32)
    )
    score_path = (
        OUTPUT_DIR / f"rr5_attention_scores_{split}_{IMPRESSION_TAG}.parquet"
    )
    scored.write_parquet(score_path)
    print("Saved fallback scores:", score_path)
    print(
        f"Scored with RR_5: branches "
        f"{[name for name, on in checkpoint['config'].items() if on]}, "
        f"loss {checkpoint['loss']}"
    )
    return scored


## 2.3 Join Scores to Candidates

Load the RR_5 score file when it covers the selected candidates; otherwise run the fallback
scorer. The generator cutoff stored in the score file must match the candidate cutoff.


In [6]:
SCORES_REUSED = False


def load_or_score():
    global SCORES_REUSED
    if REUSE_SCORES and RERANKER_SCORE_PATH.exists():
        scores = (
            pl.scan_parquet(RERANKER_SCORE_PATH)
            .filter(pl.col("impression_uid").is_in(selected_uids.to_list()))
            .select("impression_uid", "news_code", "cutoff", "score")
            .collect()
        )
        missing = candidates.join(
            scores.select("impression_uid", "news_code"),
            on=["impression_uid", "news_code"],
            how="anti",
        )
        if missing.height == 0:
            SCORES_REUSED = True
            print("Reusing RR_5 scores:", RERANKER_SCORE_PATH)
            return scores
        print(
            f"RR_5 score file is missing {missing.height:,} selected candidates; "
            "rescoring with the checkpoint."
        )
    elif REUSE_SCORES:
        print("RR_5 score file not found; scoring with the checkpoint.")
    return score_candidates(TARGET_SPLIT, candidates)


scores = load_or_score().with_columns(pl.col("news_code").cast(pl.Int64))
scored_candidates = candidates.join(
    scores, on=["impression_uid", "news_code"], how="left", suffix="_scored"
)
if scored_candidates["score"].null_count():
    raise ValueError("Some selected candidates have no reranker score")
if (
    scored_candidates["cutoff"] != scored_candidates["cutoff_scored"]
).any():
    raise ValueError("The score file and candidate file disagree on generator cutoffs")
scored_candidates = scored_candidates.drop("cutoff_scored")
print(
    f"Scored candidates: {scored_candidates.height:,} rows, "
    f"{scored_candidates['impression_uid'].n_unique():,} impressions"
)


Reusing RR_5 scores: C:\Users\Yong Rui\BT4222-Group-6-\5_Models\5_02_Reranking\artifacts\branch_interaction_generated\branch_plus_metadata__softmax_plus_lambda_ndcg__generated\generated_validation_scores.parquet


Scored candidates: 5,000,000 rows, 10,000 impressions


# 3. Form the Final Top-K Lists

Two orderings of the same generated candidates are compared:

- `generator_order`: the generator's cutoff order, truncated to K.
- `rr5_attention_reranker`: the RR_5 score descending, `news_code` ascending on ties,
  truncated to K.


In [7]:
generator_top_k = (
    candidates.filter(pl.col("cutoff") <= K)
    .with_columns((-pl.col("cutoff").cast(pl.Float64)).alias("score"))
    .select("impression_uid", "cutoff", "news_code", "score")
)

reranker_top_k = (
    scored_candidates.sort(
        ["impression_uid", "score", "news_code"], descending=[False, True, False]
    )
    .with_columns(
        pl.int_range(1, pl.len() + 1)
        .over("impression_uid")
        .cast(pl.Int16)
        .alias("cutoff")
    )
    .filter(pl.col("cutoff") <= K)
    .select("impression_uid", "cutoff", "news_code", "score")
)

for name, frame in (
    ("generator_order", generator_top_k),
    ("rr5_attention_reranker", reranker_top_k),
):
    top_shape = frame.group_by("impression_uid").agg(
        pl.len().alias("rows"),
        pl.col("cutoff").min().alias("first"),
        pl.col("cutoff").max().alias("last"),
        pl.col("cutoff").n_unique().alias("ranks"),
        pl.col("news_code").n_unique().alias("articles"),
    )
    bad = top_shape.filter(
        (pl.col("rows") != K)
        | (pl.col("first") != 1)
        | (pl.col("last") != K)
        | (pl.col("ranks") != K)
        | (pl.col("articles") != K)
    )
    if bad.height:
        raise ValueError(f"{name} has an invalid Top-K shape for {bad.height} impressions")
print("Generator Top-K rows:", generator_top_k.height)
print("Reranker Top-K rows:", reranker_top_k.height)


Generator Top-K rows: 100000
Reranker Top-K rows: 100000


# 4. Evaluate the Pipeline

Three blocks are reported for every ordering:

- **All `cg_evaluation.py` metrics at `M = K`**: pooled and impression-mean recall, hit rate,
  long-tail recall, cold-article recall, novelty, catalogue coverage and personalisation.
- **MRR@K**: reciprocal rank of the first known click in the final Top-K list.
- **nDCG@K**: binary-click nDCG over the final Top-K list.

`generator_order` is the pipeline without the reranker. It is a reference row, not a separate
model selection.


In [8]:
evaluator = CandidateEvaluator(root=REPLAY_ROOT, split=TARGET_SPLIT)

CG_METRICS = [
    "recall_pooled",
    "recall_impression_mean",
    "hit_rate",
    "long_tail_recall",
    "cold_article_recall",
    "novelty_bits",
    "catalogue_coverage",
    "personalisation",
]


def evaluate_pipeline(top_k, model_name):
    cg = evaluator.evaluate(
        top_k.select("impression_uid", "cutoff", "news_code"),
        model_name=model_name,
        m_values=(K,),
        impression_uids=selected_uids,
    )
    cg_wide = (
        cg.pivot(on="metric", index=["model", "split", "m"], values="value")
        .rename({"m": "k"})
        .with_columns(pl.lit(n_selected).alias("impressions"))
    )
    rr = evaluate_retrieved_candidates(
        top_k.select("impression_uid", "news_code", "score"),
        feature_run=RUN,
        split=TARGET_SPLIT,
        score_column="score",
        model_name=model_name,
        k_values=(K,),
    ).select(
        "model",
        pl.col("impressions").alias("clicked_impressions"),
        "mrr",
        pl.col(f"ndcg_at_{K}"),
    )
    return cg_wide.join(rr, on="model", how="left")


results = pl.concat(
    [
        evaluate_pipeline(generator_top_k, "generator_order"),
        evaluate_pipeline(reranker_top_k, "rr5_attention_reranker"),
    ],
    how="diagonal_relaxed",
)
ordered = results.select(
    "model",
    "split",
    "k",
    "impressions",
    "clicked_impressions",
    f"ndcg_at_{K}",
    "mrr",
    *CG_METRICS,
).sort(f"ndcg_at_{K}", descending=True)
display(ordered)


model,split,k,impressions,clicked_impressions,ndcg_at_10,mrr,recall_pooled,recall_impression_mean,hit_rate,long_tail_recall,cold_article_recall,novelty_bits,catalogue_coverage,personalisation
str,str,i64,i32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""generator_order""","""validation""",10,10000,10000,0.222394,0.199035,0.313491,0.356991,0.4238,0.0,0.0,9.370233,0.001893,0.749435
"""rr5_attention_reranker""","""validation""",10,10000,10000,0.171937,0.156179,0.257549,0.285708,0.3532,0.009587,0.0,11.363239,0.010665,0.814199


# 5. Save Outputs


In [9]:
final_top_k_path = (
    OUTPUT_DIR
    / f"{BASE_MODEL_NAME}_{TARGET_SPLIT}_top{K}_{IMPRESSION_TAG}.parquet"
)
reranker_top_k.write_parquet(final_top_k_path)
print("Final Top-K saved to:", final_top_k_path)

results_path = RESULTS / (
    f"pipeline_{BASE_MODEL_NAME}_{TARGET_SPLIT}_{IMPRESSION_TAG}.csv"
)
ordered.write_csv(results_path)
print("Metrics saved to:", results_path)

metadata_path = RESULTS / (
    f"pipeline_{BASE_MODEL_NAME}_{TARGET_SPLIT}_{IMPRESSION_TAG}.json"
)
metadata_path.write_text(
    json.dumps(
        {
            "pipeline": "combined_content_two_tower_rolling_popularity + RR_5 attention reranker",
            "split": TARGET_SPLIT,
            "k": K,
            "eval_impressions_requested": EVAL_IMPRESSIONS,
            "eval_impressions": int(n_selected),
            "seed": SEED,
            "candidate_path": str(CANDIDATE_PATH),
            "rr5_run_tag": RR5_RUN_TAG,
            "rr5_score_path": str(RERANKER_SCORE_PATH),
            "rr5_model_dir": str(RERANKER_DIR),
            "scores_reused": bool(SCORES_REUSED),
            "metrics": {
                row["model"]: {
                    "ndcg_at_k": row[f"ndcg_at_{K}"],
                    "mrr": row["mrr"],
                }
                for row in ordered.iter_rows(named=True)
            },
        },
        indent=2,
    )
)
print("Metadata saved to:", metadata_path)


Final Top-K saved to: C:\Users\Yong Rui\BT4222-Group-6-\5_Models\5_03_Pipeline_Evaluation\artifacts\attention_reranker_generated_validation_top10_10000imp.parquet
Metrics saved to: C:\Users\Yong Rui\BT4222-Group-6-\results\pipeline_attention_reranker_generated_validation_10000imp.csv
Metadata saved to: C:\Users\Yong Rui\BT4222-Group-6-\results\pipeline_attention_reranker_generated_validation_10000imp.json


# 6. Notes and Limitations

- The candidate generator outputs cutoffs 1-200 from rolling popularity and 201-500 from the
  content two-tower. The RR_5 reranker receives all 500 and returns the final Top-K.
- RR_5 scores are reused when `generated_<split>_scores.parquet` covers the selected
  candidates. Otherwise the fallback loads the RR_5 checkpoint, preprocessor and replayed
  features and scores the candidates itself. Nothing else is retrained.
- The RR_5 training sample is controlled in `RR_5_generated_candidates.ipynb` by
  `TRAIN_IMPRESSIONS`. This notebook only selects a saved run through `RR5_RUN_TAG`. To
  compare sample sizes, run RR_5 once per size with a distinct `RUN_TAG`, then change
  `RR5_RUN_TAG` here.
- All metrics are known-click metrics. A retrieved article that was never shown has unknown
  relevance and can never count as a click, so the final list is partly rewarded for matching
  what the logging policy showed. Promoting unshown content candidates lowers the all-retrieved
  score even when the ranking is reasonable.
- `cg_evaluation.py` metrics at `M = K` describe the final recommendation list, not the
  500-candidate pool. `catalogue_coverage` and `personalisation` therefore measure the final
  Top-K, not retrieval depth.
- For a test run, run the CG notebook with `TARGET_SPLIT = "test"` first, then either run the
  RR_5 test section or let this notebook's fallback score the test candidates. The fallback
  replays test features, which is the slow path.
